# V7 — Route Sampling Interval Validation

All data/crowd-model validations required for the MVP are complete.

The only remaining code validation is route-sampling stability, which requires real walking route geometry.

Place:

`routes_for_validation.geojson`

in the notebook directory.

If the file is absent, the cell prints `SKIPPED` instead of failing.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path.cwd()
OUT = BASE / "validation_outputs_v3"
OUT.mkdir(exist_ok=True)

ROUTES = BASE / "routes_for_validation.geojson"

if not ROUTES.exists():
    print("SKIPPED V7: routes_for_validation.geojson is not available yet.")
    print("Use real walking route LineStrings from the routing service; do not create synthetic routes for this validation.")
else:
    gj = json.loads(ROUTES.read_text())
    valid_features = []

    for i, feature in enumerate(gj.get("features", [])):
        geometry = feature.get("geometry", {})
        if geometry.get("type") == "LineString":
            valid_features.append({
                "route_id": feature.get("properties", {}).get("route_id", f"route_{i+1}"),
                "coordinate_count": len(geometry.get("coordinates", []))
            })

    check = pd.DataFrame(valid_features)

    if check.empty:
        print("SKIPPED V7: no valid LineString route features were found.")
    else:
        check.to_csv(OUT / "V7_route_file_structure_check.csv", index=False)
        display(check)

        print()
        print("Run the backend's final route-scoring function on each route using:")
        print("25 m, 50 m, 75 m, and 100 m sampling intervals.")
        print()
        print("Compare for each route:")
        print("- no_data_pct")
        print("- p75_crowd_exposure_score")
        print("- maximum_crowd_exposure_score")
        print("- pct_above_preference")
        print()
        print("Keep 50 m if it remains materially stable against the 25 m reference.")
        print("The final backend scoring function must use the V3 spatial method: 300 m max radius + normalised 1/d weighting.")